In [3]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import algoritmos_clasificacion as t
import importlib
importlib.reload(t)

# 1. Cargar dataset
S = t.cargarDataset("data_dia_c")
# 2. Atributos y variable objetivo
# X = S[["hum_carrasco", "hum_salto", "tmax_carrasco", "tmin_carrasco",
#        "tmax_salto", "tmin_salto", "amplitud_carrasco", "amplitud_salto", ""]]
#print(S.columns)
# X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto","amplitud_carrasco",
# "amplitud_salto","tprom_carrasco","tprom_salto","dir_viento_carrasco_prom","int_viento_carrasco_prom",
# "int_viento_carrasco_max","int_racha_viento_carrasco_prom","int_racha_viento_carrasco_max","dir_viento_salto_prom",
# "int_viento_salto_prom","int_viento_salto_max","int_racha_viento_salto_prom","int_racha_viento_salto_max","vis_carrasco_prom",
# "vis_carrasco_max","vis_carrasco_min","vis_salto_prom","vis_salto_max","vis_salto_min"]]

X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto","amplitud_carrasco","amplitud_salto",
"dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
"dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom","vis_carrasco_prom","vis_salto_prom"]]

targets = ["dia_lluvioso", "mes_lluvioso"]

for target in targets:
    print(f"\n=== Evaluando target: {target} ===")
    y = S[target]
    y = y.map({1: "si", 0: "no"})

    # 3. Train/Test split (80% - 20%), estratificado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # 4. Configurar validación cruzada
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)

    # 5. Comparar criterios
    criterios = ["gain", "gini", "gain_ratio"]
    resultados = {}

    for criterio in criterios:
        print(f"\n=== Evaluando criterio: {criterio.upper()} ===")
        
        # Crear modelo
        model = t.ID3Classifier(min_info_gain=0.003, criterio=criterio)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf) #default scoring accuracy
        resultados[criterio] = cv_scores
        
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio (cada fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Accuracy
        acc_test = accuracy_score(y_test, y_pred)
        
        # F1-score
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si') #pos_label='si' para la clase positiva
        
        # Recall
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred, labels=['si', 'no'])

        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)
        
    # 6. Resumen final
    print("\nRESUMEN DE COMPARACIÓN (Accuracy promedio cross-val ± desviación)")
    for criterio, scores in resultados.items():
        print(f"{criterio.upper():<12} : {scores.mean():.4f} ± {scores.std():.4f}")

    # 7. Identificar mejor criterio
    mejor_criterio = max(resultados.keys(), key=lambda k: resultados[k].mean())
    print(f"\nMejor criterio: {mejor_criterio.upper()}") #seguyn accuracy promedio (SOLO)

#matriz
      #pred si  #pred no
#   si [TP   |   FN]
#   no [FP   |   TN]


=== Evaluando target: dia_lluvioso ===

=== Evaluando criterio: GAIN ===
Scores por fold: [np.float64(0.7408), np.float64(0.7374), np.float64(0.7432), np.float64(0.7392), np.float64(0.7392)]
Accuracy promedio (cada fold): 0.7400 ± 0.0020
Accuracy en test: 0.7392
F1-score en test: 0.1521
Recall en test: 0.1056
Matriz de confusión:
[[  95  805]
 [ 254 2906]]

=== Evaluando criterio: GINI ===
Scores por fold: [np.float64(0.7408), np.float64(0.7374), np.float64(0.7432), np.float64(0.7392), np.float64(0.7392)]
Accuracy promedio (cada fold): 0.7400 ± 0.0020
Accuracy en test: 0.7392
F1-score en test: 0.1521
Recall en test: 0.1056
Matriz de confusión:
[[  95  805]
 [ 254 2906]]

=== Evaluando criterio: GAIN_RATIO ===
Scores por fold: [np.float64(0.7365), np.float64(0.742), np.float64(0.7386), np.float64(0.7355), np.float64(0.7392)]
Accuracy promedio (cada fold): 0.7384 ± 0.0023
Accuracy en test: 0.7379
F1-score en test: 0.1321
Recall en test: 0.0900
Matriz de confusión:
[[  81  819]
 [ 245 29

In [5]:
# Lista de valores a probar para min_info_gain
min_info_values = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006]

criterio = "gini"
resultados_gini = {}

for target in targets:
    print(f"\n=== Evaluando target: {target} ===")
    y = S[target]
    y = y.map({1: "si", 0: "no"})

    # 3. Train/Test split (80% - 20%), estratificado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # 4. Configurar validación cruzada
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)
    for min_gain in min_info_values:
        print(f"\n=== Evaluando GINI con min_info_gain={min_gain} ===")
        
        # Crear modelo con el valor actual de min_info_gain
        model = t.ID3Classifier(min_info_gain=min_gain, criterio=criterio)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
        resultados_gini[min_gain] = cv_scores
        
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Accuracy
        acc_test = accuracy_score(y_test, y_pred)
        # F1-score
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
        # Recall
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
        
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)

    # Resumen final de GINI para los distintos min_info_gain
    print("\nRESUMEN GINI (Accuracy promedio CV ± desviación)")
    for min_gain, scores in resultados_gini.items():
        print(f"min_info_gain={min_gain:<6} : {scores.mean():.4f} ± {scores.std():.4f}")



=== Evaluando target: dia_lluvioso ===

=== Evaluando GINI con min_info_gain=0.001 ===
Scores por fold: [np.float64(0.7408), np.float64(0.7374), np.float64(0.7432), np.float64(0.7392), np.float64(0.7392)]
Accuracy promedio CV: 0.7400 ± 0.0020
Accuracy en test: 0.7392
F1-score en test: 0.1521
Recall en test: 0.1056
Matriz de confusión:
[[  95  805]
 [ 254 2906]]

=== Evaluando GINI con min_info_gain=0.002 ===
Scores por fold: [np.float64(0.7408), np.float64(0.7374), np.float64(0.7432), np.float64(0.7392), np.float64(0.7392)]
Accuracy promedio CV: 0.7400 ± 0.0020
Accuracy en test: 0.7392
F1-score en test: 0.1521
Recall en test: 0.1056
Matriz de confusión:
[[  95  805]
 [ 254 2906]]

=== Evaluando GINI con min_info_gain=0.003 ===
Scores por fold: [np.float64(0.7408), np.float64(0.7374), np.float64(0.7432), np.float64(0.7392), np.float64(0.7392)]
Accuracy promedio CV: 0.7400 ± 0.0020
Accuracy en test: 0.7392
F1-score en test: 0.1521
Recall en test: 0.1056
Matriz de confusión:
[[  95  805]